<a href="https://colab.research.google.com/github/Lion-eni/field2map/blob/main/Field2Map_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Field2Map V1
## From field photos to mapped geographic data

Field2Map is a beginner-friendly tool that turns photographs into geographic data by reading location information already stored in a photo's EXIF metadata.

### V1 workflow
**Upload photographs → extract GPS + date/time → validate → results table → interactive map → export**

> **Important:** Field2Map does not guess a photo's location. It only reports GPS information actually available in the image metadata.


## 1. Project Setup

This is a simple data pipeline:

**Photos → EXIF metadata → GPS extraction → validation → results table → interactive map → export**

V1 deliberately stays small.

The goal is to make the foundation reliable first.


In [ ]:
# Install the packages Field2Map needs
!pip -q install exifread pandas folium


## 2. Import Libraries

- **ExifRead** reads EXIF metadata from photographs.
- **Pandas** organizes the extracted information into a table.
- **Folium** creates the interactive map.
- **Google Colab files** handles uploads and downloads.

After a Colab runtime restart, this installation cell needs to be run again before the engine cell.


##  Field2Map engine

The next cell contains the main application engine.

It reads EXIF, extracts GPS, converts coordinates to decimal degrees, extracts date/time and camera information, validates coordinates, creates the results table, and generates the map.

Run this cell after the installation cell.


In [ ]:
import os
import html
import exifread
import pandas as pd
import folium

from google.colab import files
from IPython.display import display


# ------------------------------------------------------------
# Field2Map V1 — configuration
# ------------------------------------------------------------
APP_NAME = "Field2Map V1"
OUTPUT_CSV = "field2map_results.csv"
OUTPUT_MAP = "field2map_map.html"


# ------------------------------------------------------------
# EXIF helpers
# ------------------------------------------------------------
def read_exif(filename):
    try:
        with open(filename, "rb") as image_file:
            return exifread.process_file(image_file, details=False)
    except Exception as error:
        return {"_error": str(error)}


def rational_to_float(value):
    try:
        return float(value.num) / float(value.den)
    except AttributeError:
        return float(value)


def convert_to_decimal(coordinate):
    degrees = rational_to_float(coordinate.values[0])
    minutes = rational_to_float(coordinate.values[1])
    seconds = rational_to_float(coordinate.values[2])
    return degrees + (minutes / 60) + (seconds / 3600)


def get_gps_coordinates(tags):
    latitude = tags.get("GPS GPSLatitude")
    latitude_ref = tags.get("GPS GPSLatitudeRef")
    longitude = tags.get("GPS GPSLongitude")
    longitude_ref = tags.get("GPS GPSLongitudeRef")

    if not latitude or not latitude_ref or not longitude or not longitude_ref:
        return None, None

    try:
        lat = convert_to_decimal(latitude)
        lon = convert_to_decimal(longitude)

        if str(latitude_ref).upper() == "S":
            lat = -lat
        if str(longitude_ref).upper() == "W":
            lon = -lon

        return lat, lon
    except Exception:
        return None, None


def get_tag_text(tags, key):
    value = tags.get(key)
    return str(value) if value is not None else ""


def split_datetime(value):
    if not value:
        return "", ""

    raw = str(value).strip()

    if " " in raw:
        date_part, time_part = raw.split(" ", 1)
        return date_part, time_part

    return raw, ""


# ------------------------------------------------------------
# Process one image
# ------------------------------------------------------------
def process_image(filename):
    tags = read_exif(filename)

    if "_error" in tags:
        return {
            "photo": filename,
            "latitude": None,
            "longitude": None,
            "gps_status": "GPS unavailable",
            "date_taken": "",
            "time_taken": "",
            "camera_make": "",
            "camera_model": "",
            "image_width": "",
            "image_height": "",
            "validation": "Image could not be read",
            "error": tags["_error"],
        }

    lat, lon = get_gps_coordinates(tags)
    raw_datetime = get_tag_text(tags, "EXIF DateTimeOriginal")
    date_taken, time_taken = split_datetime(raw_datetime)

    return {
        "photo": filename,
        "latitude": lat,
        "longitude": lon,
        "gps_status": "GPS found" if lat is not None and lon is not None else "GPS unavailable",
        "date_taken": date_taken,
        "time_taken": time_taken,
        "camera_make": get_tag_text(tags, "Image Make"),
        "camera_model": get_tag_text(tags, "Image Model"),
        "image_width": get_tag_text(tags, "EXIF ExifImageWidth"),
        "image_height": get_tag_text(tags, "EXIF ExifImageLength"),
        "validation": "",
        "error": "",
    }


# ------------------------------------------------------------
# Validate results
# ------------------------------------------------------------
def validate_results(df):
    df = df.copy()

    df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
    df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")

    valid_coordinates = (
        df["latitude"].between(-90, 90, inclusive="both")
        & df["longitude"].between(-180, 180, inclusive="both")
    )

    df.loc[~valid_coordinates, ["latitude", "longitude"]] = None

    usable_gps = df["latitude"].notna() & df["longitude"].notna()

    df["gps_status"] = usable_gps.map({
        True: "GPS found",
        False: "GPS unavailable"
    })

    df["validation"] = usable_gps.map({
        True: "Valid GPS coordinates",
        False: "No usable GPS coordinates"
    })

    return df


# ------------------------------------------------------------
# Interactive map
# ------------------------------------------------------------
def create_map(df):
    mapped = df[df["latitude"].notna() & df["longitude"].notna()].copy()

    if mapped.empty:
        return None

    center_lat = mapped["latitude"].mean()
    center_lon = mapped["longitude"].mean()

    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=12,
        control_scale=True
    )

    for _, row in mapped.iterrows():
        photo_name = html.escape(str(row["photo"]))
        date_taken = html.escape(str(row["date_taken"])) if row["date_taken"] else "Not available"
        time_taken = html.escape(str(row["time_taken"])) if row["time_taken"] else "Not available"

        popup_html = f"""
        <b>Photo:</b> {photo_name}<br>
        <b>Latitude:</b> {row["latitude"]:.6f}<br>
        <b>Longitude:</b> {row["longitude"]:.6f}<br>
        <b>Date:</b> {date_taken}<br>
        <b>Time:</b> {time_taken}<br>
        <b>GPS:</b> {row["gps_status"]}
        """

        folium.Marker(
            location=[row["latitude"], row["longitude"]],
            popup=folium.Popup(popup_html, max_width=350),
            tooltip=photo_name
        ).add_to(m)

    return m


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------
def generate_summary(df):
    total = len(df)
    gps_found = int((df["gps_status"] == "GPS found").sum())
    gps_missing = total - gps_found
    gps_percentage = (gps_found / total * 100) if total else 0

    return {
        "total_photos": total,
        "gps_found": gps_found,
        "gps_missing": gps_missing,
        "gps_success_rate": round(gps_percentage, 1),
    }


print(f"{APP_NAME} engine loaded successfully.")
print("Ready to process photographs.")


##  Upload photographs


Ideally use photos taken directly with a phone or camera that had location services enabled.

Avoid screenshots or copies from WhatsApp/social media for the GPS test because those copies may have had EXIF metadata removed.

Run the next cell and Colab will show **Choose Files**.


In [ ]:
# Upload photographs
uploaded = files.upload()

if not uploaded:
    raise ValueError("No photographs were uploaded.")

print(f"{len(uploaded)} photograph(s) uploaded.")
for filename in uploaded.keys():
    print(f" - {filename}")


 Process the photographs

Field2Map sends each uploaded photo through the engine.

If GPS exists, it reports the coordinates.

If GPS is missing, it reports that honestly. Field2Map does not guess a location.


In [ ]:
# Process photographs and display the Field2Map results

results = [process_image(filename) for filename in uploaded.keys()]

df = pd.DataFrame(results)
df = validate_results(df)
summary = generate_summary(df)

print("\n" + "=" * 60)
print("FIELD2MAP RESULTS")
print("=" * 60)
print(f"Total photos:       {summary['total_photos']}")
print(f"GPS found:          {summary['gps_found']}")
print(f"GPS unavailable:    {summary['gps_missing']}")
print(f"GPS success rate:   {summary['gps_success_rate']}%")

display(df[[
    "photo",
    "latitude",
    "longitude",
    "gps_status",
    "date_taken",
    "time_taken",
    "camera_make",
    "camera_model",
    "validation"
]])


## Review and Validate Results

### Latitude and longitude
These are the geographic coordinates extracted from the photo's EXIF metadata.

### GPS status
- **GPS found** — usable coordinates were extracted.
- **GPS unavailable** — usable coordinates were not available.

### Date and time
These come from the photo's EXIF `DateTimeOriginal` field when available.

### Validation
Field2Map checks whether:
- latitude is between **-90 and 90**
- longitude is between **-180 and 180**

This is basic quality control. It does not prove that the camera's recorded location is scientifically correct.


## Generate the interactive map

Every photo with valid coordinates becomes a point on the map.

Click a point to see the photo filename, coordinates, date, time, and GPS status.


In [ ]:
# Create and display the interactive map

field2map = create_map(df)

if field2map is None:
    print("No valid GPS coordinates were found, so a map cannot be generated.")
else:
    display(field2map)
    field2map.save(OUTPUT_MAP)
    print(f"Interactive map saved as: {OUTPUT_MAP}")


##  Download the interactive map

Field2Map saves the interactive map as an **HTML file**.

For V1, this file can be downloaded and opened in a web browser.

### Important
A downloaded HTML map is not automatically a permanent public URL.


In [ ]:
# Download the interactive HTML map

if os.path.exists(OUTPUT_MAP):
    files.download(OUTPUT_MAP)
else:
    print("No map file was created because no valid GPS coordinates were available.")


## Export the geographic dataset

The results table is saved as CSV.

CSV files can be opened in Excel, Google Sheets, R, Python, QGIS, ArcGIS, and other data-analysis tools.


In [ ]:
# Save and download the Field2Map dataset

df.to_csv(OUTPUT_CSV, index=False)

print(f"Dataset saved as: {OUTPUT_CSV}")
files.download(OUTPUT_CSV)
